In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [ ]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.7,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

In [ ]:
from langchain.tools import tool
import requests
import json

## Tool for accessing open-source API tool. In this case, it is space travel news
@tool
def get_space_news(n:int=1):
    """
    Returns news about space travel from SpaceflightNews API.
    """
    url = "https://spaceflightnewsapi.net/"
    params = {
        "count": n
    }
    response = requests.get(url, params=params)
    resp_dict = json.loads(response.text)
    facts_list = resp_dict.get("data", [])
    facts = "\n".join([f"{i+1}. {fact}\n" for i, fact in enumerate(facts_list)])
    return facts

## Tool for getting embeddings. I used the small text embedding model introduced in class.
## will update for this to get something useful instead of just getting embedding
'''@tool
def get_embedding(text, model="text-embedding-3-small"):
    """
    semantic embedding using text-embedding-3-small model
    """
    text = text.replace("\n", " ")
    return client.embeddings.create(input=text, model=model).data[0].embedding'''

@tool
def simple_function_gravity(mass:float):
    """
    simple function that gets force of gravity earth exerts on you at sea level given your mass
    """
    F_grav = mass*5.98*10**24*6.67*10**-11/(6.38*10^6)**2
    F_grav = str(round(F_grav))
    output = f"{F_grav} Newtons is the force of gravity earth exerts on you at sea level"
    return output
    
    
tools = [get_space_news,simple_function_gravity]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [ ]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [24]:
from langchain_core.messages import SystemMessage


def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You can get space travel news, calculate force of gravity at sea level given mass, or produce embedding for a phrase."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

In [25]:
from langchain_core.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [26]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

In [27]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

In [28]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage(content="Tell me 3 space travel news."), HumanMessage(content="creat embedding for this sentence: 'hello world'."), HumanMessage(content="I weigh 1 kg. What is force of gravity on me at sea level?")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    m.pretty_print()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)